In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.dbutils import *
from delta.tables import DeltaTable

In [0]:
# ============================================================
# SILVER TO GOLD NOTEBOOK
# Builds star schema from silver cleaned data
# All gold tables are external Delta tables in Unity Catalog
# ============================================================

STORAGE_ACCOUNT = "weatherdatalake"
CONTAINER       = "weather-data"
CATALOG         = "weather_catalog"
SILVER_SCHEMA   = "silver"
GOLD_SCHEMA     = "gold"

SILVER_TABLE    = f"{CATALOG}.{SILVER_SCHEMA}.weather_hourly_cleaned"
GOLD_PATH_BASE  = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/open_meteo/star_schema"



start_date = dbutils.widgets.get("p_start_date")
end_date   = dbutils.widgets.get("p_end_date")

print(f"Building gold layer for: {start_date} to {end_date}")

Building gold layer for:  to 


In [0]:
# Read full silver table — no date filter
# Gold MERGE handles deduplication so filtering by date is unnecessary
df_silver = spark.table(SILVER_TABLE)

print(f"Silver rows loaded: {df_silver.count()}")
print("Date range in silver:")
df_silver.agg(
    min("observation_date").alias("min_date"),
    max("observation_date").alias("max_date")
).show()
df_silver.show(3)

Silver rows loaded: 240
Date range in silver:
+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2024-01-02|2024-01-02|
+----------+----------+

+---------+---------+---------+---------+--------------+---------------------+------------------+-------------------+-------------+----------------------+---------------------+----------+----------------+-------+-----------+------------+---------------+-------------+-------------+------------------+--------------------+----------------+----------------+--------------------+--------------+
|city_name| latitude|longitude|elevation|      timezone|timezone_abbreviation|utc_offset_seconds|   observation_time|temperature_c|apparent_temperature_c|relative_humidity_pct|dewpoint_c|precipitation_mm|rain_mm|snowfall_cm|weather_code|cloud_cover_pct|windspeed_kmh|windgusts_kmh|wind_direction_deg|surface_pressure_hpa|observation_date|observation_hour| ingestion_timestamp|ingestion_date|
+---------+---------+---------+---------+--------

In [0]:
# ============================================================
# DIM_DATE — time dimension
# One row per hour across the date range
# ============================================================

df_dim_date = df_silver.select(
    col("observation_time"),
    col("observation_date"),
    col("observation_hour")
).distinct() \
.withColumn("date_key",        date_format(col("observation_time"), "yyyyMMddHH").cast("long")) \
.withColumn("year",            year(col("observation_date"))) \
.withColumn("month",           month(col("observation_date"))) \
.withColumn("month_name",      date_format(col("observation_date"), "MMMM")) \
.withColumn("day",             dayofmonth(col("observation_date"))) \
.withColumn("day_of_week",     dayofweek(col("observation_date"))) \
.withColumn("day_name",        date_format(col("observation_date"), "EEEE")) \
.withColumn("week_of_year",    weekofyear(col("observation_date"))) \
.withColumn("quarter",         quarter(col("observation_date"))) \
.withColumn("is_weekend",      when(dayofweek(col("observation_date")).isin(1,7), True).otherwise(False)) \
.withColumn("time_of_day",     
    when(col("observation_hour").between(6, 11),  lit("Morning"))
    .when(col("observation_hour").between(12, 17), lit("Afternoon"))
    .when(col("observation_hour").between(18, 21), lit("Evening"))
    .otherwise(lit("Night"))
) \
.select(
    "date_key", "observation_date", "observation_time",
    "year", "month", "month_name", "day", "day_of_week",
    "day_name", "week_of_year", "quarter", "observation_hour",
    "time_of_day", "is_weekend"
)

print(f"dim_date rows: {df_dim_date.count()}")
df_dim_date.show(5)

dim_date rows: 24
+----------+----------------+-------------------+----+-----+----------+---+-----------+--------+------------+-------+----------------+-----------+----------+
|  date_key|observation_date|   observation_time|year|month|month_name|day|day_of_week|day_name|week_of_year|quarter|observation_hour|time_of_day|is_weekend|
+----------+----------------+-------------------+----+-----+----------+---+-----------+--------+------------+-------+----------------+-----------+----------+
|2024010203|      2024-01-02|2024-01-02 03:00:00|2024|    1|   January|  2|          3| Tuesday|           1|      1|               3|      Night|     false|
|2024010218|      2024-01-02|2024-01-02 18:00:00|2024|    1|   January|  2|          3| Tuesday|           1|      1|              18|    Evening|     false|
|2024010206|      2024-01-02|2024-01-02 06:00:00|2024|    1|   January|  2|          3| Tuesday|           1|      1|               6|    Morning|     false|
|2024010219|      2024-01-02|2024-

In [0]:
# ============================================================
# DIM_LOCATION — geography dimension
# One row per city
# ============================================================

df_dim_location = df_silver.select(
    col("city_name"),
    col("latitude"),
    col("longitude"),
    col("elevation"),
    col("timezone"),
    col("timezone_abbreviation"),
    col("utc_offset_seconds")
).distinct() \
.withColumn("location_key", 
    abs(hash(col("city_name"))).cast("long")
) \
.withColumn("continent",
    when(col("city_name").isin("New_York", "Toronto"), lit("North America"))
    .when(col("city_name").isin("London", "Paris", "Berlin"), lit("Europe"))
    .when(col("city_name").isin("Tokyo", "Mumbai", "Dubai", "Singapore"), lit("Asia"))
    .when(col("city_name") == "Sydney", lit("Oceania"))
    .otherwise(lit("Unknown"))
) \
.withColumn("climate_zone",
    when(col("latitude").between(-23.5, 23.5), lit("Tropical"))
    .when((col("latitude").between(23.5, 35)) | (col("latitude").between(-35, -23.5)), lit("Subtropical"))
    .when(col("latitude").between(35, 60) | col("latitude").between(-60, -35), lit("Temperate"))
    .otherwise(lit("Polar"))
) \
.select(
    "location_key", "city_name", "latitude", "longitude",
    "elevation", "timezone", "timezone_abbreviation",
    "utc_offset_seconds", "continent", "climate_zone"
)

print(f"dim_location rows: {df_dim_location.count()}")
df_dim_location.show(truncate=False)

dim_location rows: 10
+------------+---------+----------+----------+---------+----------------+---------------------+------------------+-------------+------------+
|location_key|city_name|latitude  |longitude |elevation|timezone        |timezone_abbreviation|utc_offset_seconds|continent    |climate_zone|
+------------+---------+----------+----------+---------+----------------+---------------------+------------------+-------------+------------+
|36787689    |Toronto  |43.690685 |-79.41174 |99.0     |America/Toronto |GMT-4                |-14400            |North America|Temperate   |
|333864585   |Singapore|1.3708259 |103.80237 |46.0     |Asia/Singapore  |GMT+8                |28800             |Asia         |Tropical    |
|1320087158  |Berlin   |52.54833  |13.407822 |37.0     |Europe/Berlin   |GMT+2                |7200              |Europe       |Temperate   |
|1918125544  |Sydney   |-33.848858|151.19551 |86.0     |Australia/Sydney|GMT+10               |36000             |Oceania     

In [0]:
# ============================================================
# DIM_WEATHER_CONDITION — WMO weather code dimension
# Maps numeric weather codes to human readable descriptions
# WMO Code Table: https://open-meteo.com/en/docs
# ============================================================

weather_codes = [
    (0,  "Clear Sky",           "Clear",      "sunny"),
    (1,  "Mainly Clear",        "Clear",      "sunny"),
    (2,  "Partly Cloudy",       "Cloudy",     "cloudy"),
    (3,  "Overcast",            "Cloudy",     "cloudy"),
    (45, "Fog",                 "Fog",        "foggy"),
    (48, "Icy Fog",             "Fog",        "foggy"),
    (51, "Light Drizzle",       "Drizzle",    "rainy"),
    (53, "Moderate Drizzle",    "Drizzle",    "rainy"),
    (55, "Dense Drizzle",       "Drizzle",    "rainy"),
    (61, "Slight Rain",         "Rain",       "rainy"),
    (63, "Moderate Rain",       "Rain",       "rainy"),
    (65, "Heavy Rain",          "Rain",       "rainy"),
    (71, "Slight Snow",         "Snow",       "snowy"),
    (73, "Moderate Snow",       "Snow",       "snowy"),
    (75, "Heavy Snow",          "Snow",       "snowy"),
    (77, "Snow Grains",         "Snow",       "snowy"),
    (80, "Slight Showers",      "Showers",    "rainy"),
    (81, "Moderate Showers",    "Showers",    "rainy"),
    (82, "Violent Showers",     "Showers",    "rainy"),
    (85, "Slight Snow Showers", "Snow",       "snowy"),
    (86, "Heavy Snow Showers",  "Snow",       "snowy"),
    (95, "Thunderstorm",        "Thunder",    "stormy"),
    (96, "Thunderstorm w/ Hail","Thunder",    "stormy"),
    (99, "Thunderstorm w/ Heavy Hail", "Thunder", "stormy")
]

schema = StructType([
    StructField("weather_code",        IntegerType(), False),
    StructField("condition_desc",      StringType(),  True),
    StructField("condition_category",  StringType(),  True),
    StructField("condition_icon",      StringType(),  True)
])

df_dim_condition = spark.createDataFrame(weather_codes, schema) \
    .withColumn("condition_key", col("weather_code").cast("long"))

print(f"dim_weather_condition rows: {df_dim_condition.count()}")
df_dim_condition.show(truncate=False)

dim_weather_condition rows: 24
+------------+-------------------+------------------+--------------+-------------+
|weather_code|condition_desc     |condition_category|condition_icon|condition_key|
+------------+-------------------+------------------+--------------+-------------+
|0           |Clear Sky          |Clear             |sunny         |0            |
|1           |Mainly Clear       |Clear             |sunny         |1            |
|2           |Partly Cloudy      |Cloudy            |cloudy        |2            |
|3           |Overcast           |Cloudy            |cloudy        |3            |
|45          |Fog                |Fog               |foggy         |45           |
|48          |Icy Fog            |Fog               |foggy         |48           |
|51          |Light Drizzle      |Drizzle           |rainy         |51           |
|53          |Moderate Drizzle   |Drizzle           |rainy         |53           |
|55          |Dense Drizzle      |Drizzle           |rai

In [0]:
# ============================================================
# FACT_WEATHER_HOURLY — central fact table
# One row per city per hour — all measurements + foreign keys
# ============================================================

df_fact = df_silver \
    .withColumn("date_key",
        date_format(col("observation_time"), "yyyyMMddHH").cast("long")
    ) \
    .withColumn("location_key",
        abs(hash(col("city_name"))).cast("long")
    ) \
    .withColumn("condition_key",
        col("weather_code").cast("long")
    ) \
    .withColumn("fact_key",
        abs(hash(concat(col("city_name"), col("observation_time").cast("string")))).cast("long")
    ) \
    .select(
        # Surrogate key
        "fact_key",
        # Foreign keys
        "date_key",
        "location_key",
        "condition_key",
        # Degenerate dimensions
        col("observation_time"),
        col("observation_date"),
        col("city_name"),
        # Measures
        col("temperature_c"),
        col("apparent_temperature_c"),
        col("relative_humidity_pct"),
        col("dewpoint_c"),
        col("precipitation_mm"),
        col("rain_mm"),
        col("snowfall_cm"),
        col("cloud_cover_pct"),
        col("windspeed_kmh"),
        col("windgusts_kmh"),
        col("wind_direction_deg"),
        col("surface_pressure_hpa"),
        # Audit
        col("ingestion_timestamp"),
        col("ingestion_date")
    )

print(f"fact_weather_hourly rows: {df_fact.count()}")
df_fact.show(5)

fact_weather_hourly rows: 240
+----------+----------+------------+-------------+-------------------+----------------+---------+-------------+----------------------+---------------------+----------+----------------+-------+-----------+---------------+-------------+-------------+------------------+--------------------+--------------------+--------------+
|  fact_key|  date_key|location_key|condition_key|   observation_time|observation_date|city_name|temperature_c|apparent_temperature_c|relative_humidity_pct|dewpoint_c|precipitation_mm|rain_mm|snowfall_cm|cloud_cover_pct|windspeed_kmh|windgusts_kmh|wind_direction_deg|surface_pressure_hpa| ingestion_timestamp|ingestion_date|
+----------+----------+------------+-------------+-------------------+----------------+---------+-------------+----------------------+---------------------+----------+----------------+-------+-----------+---------------+-------------+-------------+------------------+--------------------+--------------------+-----------

In [0]:
# ============================================================
# HELPER — write or merge external Delta table in gold layer
# ============================================================

def write_gold_table(df, table_name, path, merge_keys, partition_cols=None):
    full_table_name = f"{CATALOG}.{GOLD_SCHEMA}.{table_name}"
    full_path       = f"{GOLD_PATH_BASE}/{table_name}/"
    
    table_exists = spark.catalog.tableExists(full_table_name)
    
    if not table_exists:
        print(f"Creating: {full_table_name}")
        writer = df.write.format("delta").mode("overwrite")
        if partition_cols:
            writer = writer.partitionBy(partition_cols)
        writer.save(full_path)
        
        spark.sql(f"""
            CREATE TABLE {full_table_name}
            USING DELTA
            LOCATION '{full_path}'
        """)
        print(f"✅ Created: {full_table_name}")

    else:
        print(f"Merging: {full_table_name}")
        delta_table = DeltaTable.forPath(spark, full_path)
        
        merge_condition = " AND ".join(
            [f"target.{k} = source.{k}" for k in merge_keys]
        )
        
        delta_table.alias("target").merge(
            df.alias("source"), merge_condition
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()
        print(f"✅ Merged: {full_table_name}")

In [0]:
# Write dimension tables
write_gold_table(df_dim_date,      "dim_date",              GOLD_PATH_BASE, ["date_key"])
write_gold_table(df_dim_location,  "dim_location",          GOLD_PATH_BASE, ["location_key"])
write_gold_table(df_dim_condition, "dim_weather_condition", GOLD_PATH_BASE, ["condition_key"])

# Write fact table — partition by observation_date for performance
write_gold_table(df_fact, "fact_weather_hourly", GOLD_PATH_BASE, 
                 ["fact_key"], partition_cols=["observation_date"])

print("\n✅ ALL GOLD TABLES WRITTEN SUCCESSFULLY")

Creating: weather_catalog.gold.dim_date
✅ Created: weather_catalog.gold.dim_date
Creating: weather_catalog.gold.dim_location
✅ Created: weather_catalog.gold.dim_location
Creating: weather_catalog.gold.dim_weather_condition
✅ Created: weather_catalog.gold.dim_weather_condition
Creating: weather_catalog.gold.fact_weather_hourly
✅ Created: weather_catalog.gold.fact_weather_hourly

✅ ALL GOLD TABLES WRITTEN SUCCESSFULLY


In [0]:
# Optimize fact table — most queried table
spark.sql(f"OPTIMIZE {CATALOG}.{GOLD_SCHEMA}.fact_weather_hourly ZORDER BY (city_name, temperature_c)")
spark.sql(f"OPTIMIZE {CATALOG}.{GOLD_SCHEMA}.dim_date ZORDER BY (year, month, day)")
spark.sql(f"OPTIMIZE {CATALOG}.{GOLD_SCHEMA}.dim_location ZORDER BY (city_name)")

print("✅ All gold tables optimized")

✅ All gold tables optimized


In [0]:
# ============================================================
# VERIFY — join all tables to confirm star schema works
# ============================================================

df_check = spark.sql(f"""
    SELECT
        l.city_name,
        l.continent,
        l.climate_zone,
        d.year,
        d.month_name,
        d.day_name,
        d.time_of_day,
        c.condition_desc,
        c.condition_category,
        ROUND(AVG(f.temperature_c), 2)       AS avg_temp_c,
        ROUND(AVG(f.relative_humidity_pct), 2) AS avg_humidity,
        ROUND(SUM(f.precipitation_mm), 2)    AS total_precipitation_mm,
        ROUND(MAX(f.windspeed_kmh), 2)       AS max_windspeed_kmh
    FROM {CATALOG}.{GOLD_SCHEMA}.fact_weather_hourly f
    JOIN {CATALOG}.{GOLD_SCHEMA}.dim_location        l ON f.location_key  = l.location_key
    JOIN {CATALOG}.{GOLD_SCHEMA}.dim_date            d ON f.date_key      = d.date_key
    JOIN {CATALOG}.{GOLD_SCHEMA}.dim_weather_condition c ON f.condition_key = c.condition_key
    GROUP BY
        l.city_name, l.continent, l.climate_zone,
        d.year, d.month_name, d.day_name, d.time_of_day,
        c.condition_desc, c.condition_category
    ORDER BY l.city_name, d.year, d.month_name
""")

print(f"Star schema query returned: {df_check.count()} rows")
df_check.show(20, truncate=False)

Star schema query returned: 96 rows
+---------+---------+------------+----+----------+--------+-----------+----------------+------------------+----------+------------+----------------------+-----------------+
|city_name|continent|climate_zone|year|month_name|day_name|time_of_day|condition_desc  |condition_category|avg_temp_c|avg_humidity|total_precipitation_mm|max_windspeed_kmh|
+---------+---------+------------+----+----------+--------+-----------+----------------+------------------+----------+------------+----------------------+-----------------+
|Berlin   |Europe   |Temperate   |2024|January   |Tuesday |Night      |Overcast        |Cloudy            |3.98      |89.8        |0.0                   |13.9             |
|Berlin   |Europe   |Temperate   |2024|January   |Tuesday |Evening    |Moderate Drizzle|Drizzle           |6.3       |95.0        |0.8                   |13.5             |
|Berlin   |Europe   |Temperate   |2024|January   |Tuesday |Afternoon  |Light Drizzle   |Drizzle    